In [ ]:
%pip install importlib-resources

In [1]:
# Chit 26: Caltech-101 VGG16 feature extractor
# Step 1: Import libraries 
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from sklearn.model_selection import train_test_split



In [2]:
# Load Caltech-101
(ds,), info = tfds.load(
    "caltech101",
    split=["train"],
    as_supervised=True,
    with_info=True
)
all_names = info.features["label"].names



In [3]:
# Select five categories
selected_names = ["airplanes", "ant", "camera", "cannon", "chair"]
missing = [name for name in selected_names if name not in all_names]
if missing:
    raise ValueError(f"Categories not found: {missing}")

selected_ids = [all_names.index(name) for name in selected_names]

# Select images, resize, and assign labels 0–4
images, labels = [], []
for image, label in tfds.as_numpy(ds):
    label = int(label)
    if label in selected_ids:
        images.append(tf.image.resize(image, (224, 224)).numpy())
        labels.append(selected_ids.index(label))

X = np.asarray(images, dtype=np.float32)
y = np.asarray(labels, dtype=np.int32)

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [4]:
# Apply VGG16 preprocessing
X_train = preprocess_input(X_train.copy())
X_test = preprocess_input(X_test.copy())

# Load VGG16 without its original classifier and freeze its layers
vgg = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)
vgg.trainable = False

# Extract features using VGG16 and global average pooling
inputs = layers.Input(shape=(224, 224, 3))
features = layers.GlobalAveragePooling2D()(vgg(inputs, training=False))
feature_extractor = Model(inputs, features)

train_features = feature_extractor.predict(X_train, batch_size=16)
test_features = feature_extractor.predict(X_test, batch_size=16)

8/8 ━━━━━━━━━━━━━━━━━━━━ 18s 2s/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 2s/step


In [5]:
# Train a Softmax classifier on the extracted features
classifier = tf.keras.Sequential([
    layers.Input(shape=(train_features.shape[1],)),
    layers.Dense(len(selected_names), activation="softmax")
])
classifier.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
classifier.fit(train_features, y_train, epochs=15, batch_size=32)

Epoch 1/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - accuracy: 0.2417 - loss: 5.2056
Epoch 2/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.3667 - loss: 3.2881
Epoch 3/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.5417 - loss: 2.0878
Epoch 4/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.7083 - loss: 1.3036
Epoch 5/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8500 - loss: 0.7945
Epoch 6/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.8833 - loss: 0.5246
Epoch 7/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.9250 - loss: 0.3692
Epoch 8/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.9500 - loss: 0.2598 
Epoch 9/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.9583 - loss: 0.1769
Epoch 10/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.9667 - loss: 0.1171
Epoch 11/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.9750 - loss: 0.0659
Epoch 12/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9917 - loss: 0.0378 

In [6]:
# Evaluate on held-out test features
_, test_accuracy = classifier.evaluate(test_features, y_test, verbose=0)
print(f"Test accuracy: {test_accuracy * 100:.2f}%")

Test accuracy: 86.67%
